---
<div style="display: flex; justify-content: space-between; align-items: center; width: 100%;">
    <img src='../misc/ufpr.png' style='width: 15%;'>
    <div style="text-align: center; flex-grow: 1;">
        <h2 style='line-height: 0.5; color:#ffda77; margin-top: 50px;'>Universidade Federal do Paraná</h2>
        <h2 style='line-height: 0.5; color:#ffda77;'>LABAP</h2>
        <h6 style='line-height: 0.5; color:#ffda77;'>Laboratório de Análise de Bacias e Petrofísica</h6>
    </div>
    <img src='../misc/labap.png' style='width: 15%;'>
</div>
<br>

---

# Initial Algorithm Screening — LazyPredict

**Objective:** Quick evaluation of ~40 regression algorithms to identify candidates for the Leave-One-Well-Out (LOWO) validation process, and to investigate the impact of the Caliper (CALI) as a feature and of the IQR filter as a preprocessing step.

**Evaluation strategy:** Simple train/test split (hold-out), using well `3-BRSA-1163-SES` as the test set.

**Three scenarios evaluated:**
- **Model A** — Data without IQR, features without CALI
- **Model A_cali** — Data without IQR, features with CALI
- **Model B** — Data with IQR, features without CALI *(adopted model)*

**Note on execution:** LazyPredict results are saved to CSV after the first run. In subsequent runs, results are loaded directly from cache, avoiding the computational cost.

**Structure:**
1. Imports and settings
2. Model A — without IQR, without CALI
3. Model A_cali — without IQR, with CALI
4. Model B — with IQR, without CALI
5. Comparative analysis: impact of CALI
6. Comparative analysis: impact of IQR
7. Final decisions and configuration for LOWO

---

## 1. Imports and Settings

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from lazypredict.Supervised import LazyRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# -- Global settings --------------------------------------------------------
FEATURES        = ['GR', 'RT90', 'RHOB', 'NPHI']
FEATURES_W_CALI = ['GR', 'RT90', 'RHOB', 'NPHI', 'CALI']
TARGET          = 'DT'
TEST_WELL       = '3-BRSA-1163-SES'

# -- Directories -------------------------------------------------------------
METRICS_DIR = Path('../results/comparison/metrics')
FIGURES_DIR = Path('../results/comparison/figures')

METRICS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Features (without CALI): {FEATURES}')
print(f'Features (with CALI): {FEATURES_W_CALI}')
print(f'Test well: {TEST_WELL}')

In [ ]:
# -- Helper functions --------------------------------------------------------

def run_or_load_lazypredict(X_train, X_test, y_train, y_test, cache_path):
    """
    Runs LazyRegressor or loads results from cache if they already exist.
    Avoids unnecessary re-execution in future sessions.
    """
    cache_path = Path(cache_path)
    if cache_path.exists():
        models = pd.read_csv(cache_path, index_col=0)
        print(f'✓ Results loaded from cache: {cache_path.name}')
        print(f'  {len(models)} algorithms available.')
        return models
    
    print('Running LazyRegressor (first run — may take a while)...')
    reg = LazyRegressor(verbose=1, ignore_warnings=True, custom_metric=None)
    models, _ = reg.fit(X_train, X_test, y_train, y_test)
    models.to_csv(cache_path)
    print(f'✓ Results saved to: {cache_path.name}')
    print(f'  {len(models)} algorithms tested.')
    return models


def analyze_top_10(models):
    """Displays detailed analysis of the top 10 algorithms."""
    top_10 = models.head(min(10, len(models)))
    print(f'{"Rank":<5} {"Algorithm":<35} {"R²":>8} {"RMSE":>8} {"Time(s)":>10} {"R²/Time":>10}')
    print('-' * 80)
    for rank, (name, row) in enumerate(top_10.iterrows(), 1):
        efficiency = row['R-Squared'] / row['Time Taken'] if row['Time Taken'] > 0 else float('inf')
        print(f'{rank:<5} {name:<35} {row["R-Squared"]:>8.4f} {row["RMSE"]:>8.4f} '
              f'{row["Time Taken"]:>10.2f} {efficiency:>10.4f}')


def identify_lowo_candidates(models):
    """Identifies LOWO candidates by R² and by efficiency (R²/Time)."""
    m = models.copy()
    m['Efficiency'] = m['R-Squared'] / m['Time Taken']

    top_r2  = m.nlargest(5, 'R-Squared')
    top_eff = m.nlargest(5, 'Efficiency')

    print('Top 5 by R²:')
    for rank, (name, row) in enumerate(top_r2.iterrows(), 1):
        print(f'  {rank}. {name:<35} R²: {row["R-Squared"]:.4f}')

    print('\nTop 5 by Efficiency (R²/Time):')
    for rank, (name, row) in enumerate(top_eff.iterrows(), 1):
        print(f'  {rank}. {name:<35} Efficiency: {row["Efficiency"]:.4f}  '
              f'(R²: {row["R-Squared"]:.4f}, {row["Time Taken"]:.2f}s)')

    candidates = set(top_r2.index).union(set(top_eff.index))
    print(f'\nLOWO candidates ({len(candidates)} algorithms):')
    for name in sorted(candidates):
        row = m.loc[name]
        print(f'  • {name:<35} R²: {row["R-Squared"]:.4f}, RMSE: {row["RMSE"]:.4f}, '
              f'Time: {row["Time Taken"]:.2f}s')


print('Helper functions defined.')

## 2. Model A — Data without IQR, Features without CALI

Baseline scenario: full dataset (no IQR filter) and standard feature set [GR, RT90, RHOB, NPHI].

In [ ]:
wells_no_iqr = pd.read_csv('../data/processed/wells_no_iqr.csv').dropna()

train_A = wells_no_iqr[wells_no_iqr['Well_Name'] != TEST_WELL].copy()
test_A  = wells_no_iqr[wells_no_iqr['Well_Name'] == TEST_WELL].copy()

X_train_A = train_A[FEATURES]
y_train_A = train_A[TARGET]
X_test_A  = test_A[FEATURES]
y_test_A  = test_A[TARGET]

print(f'Dataset: {len(wells_no_iqr):,} samples | {wells_no_iqr["Well_Name"].nunique()} wells')
print(f'Train: {len(train_A):,} samples ({train_A["Well_Name"].nunique()} wells)')
print(f'Test : {len(test_A):,} samples (well: {TEST_WELL})')

> **Cache notice:** if `lazypredict_models_A.csv` already exists in `results/comparison/metrics/`, this cell loads cached results instead of re-running LazyRegressor. Delete the CSV to force a fresh run.

In [ ]:
models_A = run_or_load_lazypredict(
    X_train_A, X_test_A, y_train_A, y_test_A,
    cache_path=METRICS_DIR / 'lazypredict_models_A.csv'
)

In [ ]:
models_A.head(15)

In [ ]:
analyze_top_10(models_A)

In [ ]:
identify_lowo_candidates(models_A)

## 3. Model A_cali — Data without IQR, Features with CALI

Same dataset as Model A, now including the Caliper (CALI) as a predictor feature. The goal is to assess whether CALI contributes positively or introduces noise in DT prediction.

> **Cache notice:** if `lazypredict_models_A_cali.csv` already exists in `results/comparison/metrics/`, this cell loads cached results instead of re-running LazyRegressor. Delete the CSV to force a fresh run.

In [ ]:
# Reuses the same split (train_A / test_A) — only the features change
X_train_A_cali = train_A[FEATURES_W_CALI]
X_test_A_cali  = test_A[FEATURES_W_CALI]

models_A_cali = run_or_load_lazypredict(
    X_train_A_cali, X_test_A_cali, y_train_A, y_test_A,
    cache_path=METRICS_DIR / 'lazypredict_models_A_cali.csv'
)

In [ ]:
models_A_cali.head(15)

In [ ]:
analyze_top_10(models_A_cali)

In [ ]:
identify_lowo_candidates(models_A_cali)

## 4. Model B — Data with IQR, Features without CALI

IQR-filtered dataset (outlier removal via CALI) and standard features without CALI. This is the scenario adopted in the subsequent LOWO notebooks.

In [ ]:
wells_iqr = pd.read_csv('../data/processed/wells_iqr.csv').dropna()

train_B = wells_iqr[wells_iqr['Well_Name'] != TEST_WELL].copy()
test_B  = wells_iqr[wells_iqr['Well_Name'] == TEST_WELL].copy()

X_train_B = train_B[FEATURES]
y_train_B = train_B[TARGET]
X_test_B  = test_B[FEATURES]
y_test_B  = test_B[TARGET]

print(f'Dataset (with IQR): {len(wells_iqr):,} samples | {wells_iqr["Well_Name"].nunique()} wells')
print(f'Train: {len(train_B):,} samples ({train_B["Well_Name"].nunique()} wells)')
print(f'Test : {len(test_B):,} samples (well: {TEST_WELL})')
print(f'IQR reduction: {len(wells_no_iqr) - len(wells_iqr):,} samples '
      f'({(1 - len(wells_iqr)/len(wells_no_iqr))*100:.1f}%)')

> **Cache notice:** if `lazypredict_models_B.csv` already exists in `results/comparison/metrics/`, this cell loads cached results instead of re-running LazyRegressor. Delete the CSV to force a fresh run.

In [ ]:
models_B = run_or_load_lazypredict(
    X_train_B, X_test_B, y_train_B, y_test_B,
    cache_path=METRICS_DIR / 'lazypredict_models_B.csv'
)

In [ ]:
models_B.head(15)

In [ ]:
analyze_top_10(models_B)

In [ ]:
identify_lowo_candidates(models_B)

## 5. Comparative Analysis: Impact of CALI as a Feature

Direct comparison between Model A (without CALI) and Model A_cali (with CALI) on the same data (without IQR), evaluating the impact of including the Caliper on algorithm performance.

In [ ]:
# Build comparison table for the top 10 of Model A
comparison_cali = []
for name in models_A.head(10).index:
    if name in models_A_cali.index:
        r2_sem  = models_A.loc[name, 'R-Squared']
        r2_com  = models_A_cali.loc[name, 'R-Squared']
        rmse_sem = models_A.loc[name, 'RMSE']
        rmse_com = models_A_cali.loc[name, 'RMSE']
        comparison_cali.append({
            'Algorithm'        : name,
            'R2_out_CALI'      : r2_sem,
            'R2_with_CALI'      : r2_com,
            'Delta_R2'         : r2_sem - r2_com,
            'Degradation_R2_pct': (r2_sem - r2_com) / r2_sem * 100,
            'RMSE_out_CALI'    : rmse_sem,
            'RMSE_with_CALI'    : rmse_com,
            'Delta_RMSE'       : rmse_com - rmse_sem,
        })

df_cali = pd.DataFrame(comparison_cali).sort_values('Delta_R2', ascending=False)

# Save
df_cali.to_csv(METRICS_DIR / 'cali_impact_comparison.csv', index=False)

print('CALIPER IMPACT AS A FEATURE — TOP 10 ALGORITHMS')
print('=' * 90)
print(df_cali.to_string(index=False))
print()
print(f'Average R² degradation : {df_cali["Delta_R2"].mean():.4f} ({df_cali["Degradation_R2_pct"].mean():.2f}%)')
print(f'Average RMSE increase  : {df_cali["Delta_RMSE"].mean():.4f} µs/ft')
print(f'Algorithms that worsened: {(df_cali["Delta_R2"] > 0).sum()}/{len(df_cali)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Impact of Caliper (CALI) as a Feature — Top 10 Algorithms',
             fontsize=13, fontweight='bold')

x     = range(len(df_cali))
width = 0.35
labels = [n.replace('Regressor', '') for n in df_cali['Algorithm']]

# R²
axes[0].bar([i - width/2 for i in x], df_cali['R2_out_CALI'],
            width, label='Without CALI', color='#2ecc71', alpha=0.85)
axes[0].bar([i + width/2 for i in x], df_cali['R2_with_CALI'],
            width, label='With CALI', color='#e74c3c', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha='right')
axes[0].set_ylabel('R²')
axes[0].set_title('R²: Without CALI vs With CALI')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Degradation %
colors_deg = ['#e74c3c' if v > 0 else '#2ecc71' for v in df_cali['Degradation_R2_pct']]
bars = axes[1].barh(df_cali['Algorithm'], df_cali['Degradation_R2_pct'],
                    color=colors_deg, alpha=0.8)
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')
for bar, val in zip(bars, df_cali['Degradation_R2_pct']):
    axes[1].text(val + (0.3 if val >= 0 else -0.3), bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
axes[1].set_xlabel('R² Degradation (%)')
axes[1].set_title('Percentage Degradation When Including CALI\n(positive = CALI hurts)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'cali_impact_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 6. Comparative Analysis: Impact of the IQR Filter

Comparison between Model A (without IQR) and Model B (with IQR), both without CALI, to assess whether outlier removal via the IQR filter improves algorithm performance.

In [ ]:
comparison_iqr = []
for name in models_A.head(10).index:
    if name in models_B.index:
        r2_sem   = models_A.loc[name, 'R-Squared']
        r2_com   = models_B.loc[name, 'R-Squared']
        rmse_sem = models_A.loc[name, 'RMSE']
        rmse_com = models_B.loc[name, 'RMSE']
        comparison_iqr.append({
            'Algorithm'   : name,
            'R2_out_IQR'  : r2_sem,
            'R2_with_IQR'  : r2_com,
            'Delta_R2'    : r2_com - r2_sem,
            'RMSE_out_IQR': rmse_sem,
            'RMSE_with_IQR': rmse_com,
            'Delta_RMSE'  : rmse_sem - rmse_com,
        })

df_iqr = pd.DataFrame(comparison_iqr).sort_values('Delta_R2', ascending=False)
df_iqr.to_csv(METRICS_DIR / 'iqr_impact_comparison.csv', index=False)

print('IQR FILTER IMPACT — TOP 10 ALGORITHMS')
print('=' * 85)
print(df_iqr.to_string(index=False))
print()
print(f'Average R² improvement : +{df_iqr["Delta_R2"].mean():.4f}')
print(f'Average RMSE reduction : -{df_iqr["Delta_RMSE"].mean():.4f} µs/ft')
print(f'Algorithms that improved: {(df_iqr["Delta_R2"] > 0).sum()}/{len(df_iqr)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Impact of the IQR Filter — Top 10 Algorithms',
             fontsize=13, fontweight='bold')

x      = range(len(df_iqr))
labels = [n.replace('Regressor', '') for n in df_iqr['Algorithm']]

axes[0].plot(x, df_iqr['R2_out_IQR'], 'o-', label='Without IQR',
             linewidth=2, markersize=7, color='#e74c3c')
axes[0].plot(x, df_iqr['R2_with_IQR'], 's-', label='With IQR',
             linewidth=2, markersize=7, color='#2ecc71')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=45, ha='right')
axes[0].set_ylabel('R²')
axes[0].set_title('R²: Without IQR vs With IQR')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(x, df_iqr['RMSE_out_IQR'], 'o-', label='Without IQR',
             linewidth=2, markersize=7, color='#e74c3c')
axes[1].plot(x, df_iqr['RMSE_with_IQR'], 's-', label='With IQR',
             linewidth=2, markersize=7, color='#2ecc71')
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right')
axes[1].set_ylabel('RMSE (µs/ft)')
axes[1].set_title('RMSE: Without IQR vs With IQR')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'iqr_impact_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 7. Final Decisions and Configuration for LOWO

In [ ]:
print('=' * 70)
print('METHODOLOGICAL DECISIONS BASED ON SCREENING (LazyPredict)')
print('=' * 70)

# Statistics to support the decisions
deg_cali_media = df_cali['Degradation_R2_pct'].mean()
deg_cali_n     = (df_cali['Delta_R2'] > 0).sum()
melhora_iqr_r2 = df_iqr['Delta_R2'].mean()
melhora_iqr_rmse = df_iqr['Delta_RMSE'].mean()
n_melhora_iqr  = (df_iqr['Delta_R2'] > 0).sum()

print(f'''
1. EXCLUSION OF CALIPER (CALI) AS A FEATURE
   Result: {deg_cali_n}/{len(df_cali)} algorithms showed degradation when including CALI.
   Average degradation: {deg_cali_media:.2f}% in R².
   Decision: Features = {FEATURES}

2. ADOPTION OF THE IQR FILTER
   Result: {n_melhora_iqr}/{len(df_iqr)} algorithms improved with the IQR filter.
   Average improvement: +{melhora_iqr_r2:.4f} R² | -{melhora_iqr_rmse:.4f} µs/ft RMSE.
   Decision: Dataset = wells_iqr.csv

3. ALGORITHMS SELECTED FOR LOWO (Model B — final configuration)
''')

# Top candidates from Model B
top_B = models_B.head(10).copy()
top_B['Efficiency'] = top_B['R-Squared'] / top_B['Time Taken']
print(f'  {"Algorithm":<35} {"R²":>8} {"RMSE":>8} {"Time(s)":>10}')
print(f'  {"-"*65}')
for name, row in top_B.head(8).iterrows():
    print(f'  {name:<35} {row["R-Squared"]:>8.4f} {row["RMSE"]:>8.4f} {row["Time Taken"]:>10.2f}')

print(f'\n  Note: SVR/NuSVR discarded due to prohibitive runtime (>1200s) in LOWO with 32 wells.')
print('=' * 70)